# Import Libraries

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Read the dataset 

In [ ]:
df = pd.read_csv("youTubeSearchListChattishgarh.csv",index_col=[0])

In [ ]:
df.head()

In [ ]:
df.shape

# Check for null value count

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

# Converting Object dtype to strings

In [ ]:
df['title'] = df['title'].convert_dtypes()
df['description'] = df['description'].convert_dtypes()
df['tags'] = df['tags'].convert_dtypes()

In [ ]:
df.info()

# Dropping Null Values

In [ ]:
df = df.dropna()
df = df.reset_index(drop=True)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

# Removing tags, special characters and digits, convert to lower case

In [ ]:
import re
def pre_process(text):
    
    # lowercase
    text=text.lower()
    
    #remove tags
    text=re.sub("","",text)
    
    # remove special characters and digits
    text=re.sub("(\\d|\\W)+"," ",text)
    
    return text

df['text'] = df['title'] + df['description'] + df['tags']
df['text'] = df['text'].apply(lambda x:pre_process(x))

#show the second 'text' just for fun
df['text'][2]

In [ ]:
df.head()

# Lemmatization and Removal of Stopwords

In [ ]:
import re
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
import string
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
processed_all_documents = list()

for desc in df['text']:
    word_tokens = word_tokenize(desc) 
    
    filtered_sentence = [] 

    # Removing Stopwords
    for w in word_tokens: 
        if w not in stop_words: 
            filtered_sentence.append(w) 
    
    # Lemmetization
    lemma_word = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for w in filtered_sentence:
        word1 = wordnet_lemmatizer.lemmatize(w, pos = "n")
        word2 = wordnet_lemmatizer.lemmatize(word1, pos = "v")
        word3 = wordnet_lemmatizer.lemmatize(word2, pos = ("a"))
        lemma_word.append(word3)
    words = ' '.join(lemma_word)
    processed_all_documents.append(words)
df['Processed_text'] = processed_all_documents
df['Word_Count'] = [len(desc.split(' ')) for desc in df['Processed_text']]

wordCount_after = df['Word_Count'].sum()
print("Total Words after processing: ", wordCount_after)
print("Max word count of a Document: ", df['Word_Count'].max())
print("Mean word count of Documents: ", df['Word_Count'].mean())

In [ ]:
df.head()

# Create a vocabulary of words, 
Ignore words that appear in 85% of documents, 
Eliminate stop words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import re

def get_stop_words(stop_file_path):
    """load stop words """
    
    with open(stop_file_path, 'r', encoding="utf-8") as f:
        stopwords = f.readlines()
        stop_set = set(m.strip() for m in stopwords)
        return frozenset(stop_set)

#load a set of stop words
stopwords=get_stop_words("stopwords.txt")

#get the text column 
docs=df['Processed_text'].tolist()

#create a vocabulary of words, 
#ignore words that appear in 85% of documents, 
#eliminate stop words
cv=CountVectorizer(max_df=0.85,stop_words=stopwords)
word_count_vector=cv.fit_transform(docs)

In [ ]:
word_count_vector.shape

In [ ]:
list(cv.vocabulary_.keys())[:10]

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer=TfidfTransformer(smooth_idf=True,use_idf=True)
tfidf_transformer.fit(word_count_vector)

In [ ]:
# you only needs to do this once, this is a mapping of index to 
feature_names=cv.get_feature_names()

In [ ]:
feature_names

# Generate tf-idf for the given document

In [ ]:
i = 0
tex_list = df['Processed_text'].tolist()
tf_idf_vector_list = []
for i in range(len(tex_list)):
    tf_idf_vector=tfidf_transformer.transform(cv.transform([tex_list[i]]))
    i = i+1
    tf_idf_vector_list.append(tf_idf_vector)
    # added to list  
    if ((i%10000) == 0):
        print ('Collecting tf-idf',i)
print(len(tf_idf_vector_list))

In [ ]:
tf_idf_vector_list[0].todense()

In [ ]:
tf_idf_vector_list[0].todense().shape

# Label Generation for Supervised Learning
K-Means Clustering is done based on Processed text

In [ ]:
#Clustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

In [ ]:
document = df['Processed_text'].values.astype('U')
vectorizer = TfidfVectorizer(stop_words='english')
features = vectorizer.fit_transform(document)

In [ ]:
k=20 #Obtained from Silhouette_score
model = KMeans(n_clusters=k,init='k-means++',max_iter=100,n_init=1)
label = model.fit(features)

# Calculation of optimal value of K

In [ ]:
# from sklearn.metrics import silhouette_score

# sil = []
# kmax = 20
# sil
# # dissimilarity would not be defined for a single cluster, thus, minimum number of clusters should be 2
# for k in range(2, kmax+1):
#     kmeans = KMeans(n_clusters = k).fit(features)
#     labels = kmeans.labels_
#     sil.append(silhouette_score(features, labels, metric = 'euclidean'))
#     print("Done for k =",k)


In [ ]:
df['cluster'] = model.labels_

In [ ]:
print(model.labels_)

In [ ]:
print(model.labels_)

In [ ]:
df.head()

# Checking number of items in each cluster

In [ ]:
i = 0
for i in range(len(df['cluster'].unique())):
    count = (df.cluster == i).sum()
    print(f"Total count of cluster {i} is {count}")

# Sort the tf-idf vectors by descending order of scores

In [ ]:
def sort_coo(coo_matrix):
    tuples = zip(coo_matrix.col, coo_matrix.data)
    return sorted(tuples, key=lambda x: (x[1], x[0]), reverse=True)

In [ ]:
sorted_items_list=[]
i=0
for i in range(len(tf_idf_vector_list)):
    sorted_items=sort_coo(tf_idf_vector_list[i].tocoo())
    i = i+1
    sorted_items_list.append(sorted_items)
    # added to list  
    if ((i%10000) == 0):
        print ('Sorting Done for',i)

In [ ]:
len(sorted_items_list)

# Extract only the top n; n here is 5

In [ ]:
def extract_topn_from_vector(feature_names, sorted_items, topn=5):
    """get the feature names and tf-idf score of top n items"""
    
    #use only topn items from vector
    sorted_items = sorted_items[:topn]

    score_vals = []
    feature_vals = []
    
    # word index and corresponding tf-idf score
    for idx, score in sorted_items:
        
        #keep track of feature name and its corresponding score
        score_vals.append(round(score, 3))
        feature_vals.append(feature_names[idx])

    #create a tuples of feature,score
    #results = zip(feature_vals,score_vals)
    results= {}
    for idx in range(len(feature_vals)):
        results[feature_vals[idx]]=score_vals[idx]
    
    return results

In [ ]:
keyword_list = []
i=0
for i in range(len(sorted_items_list)):
    keywords=extract_topn_from_vector(feature_names,sorted_items_list[i],5)
    i = i+1
    keyword_list.append(keywords)
    # added to list  
    if ((i%10000) == 0):
        print ('Keywords added for',i)

In [ ]:
len(keyword_list)

In [ ]:
def getList(dict):
    return list(dict.keys())
key_list = []
i = 0
for i in range(len(keyword_list)):
    keys = getList(keyword_list[i])
    key_list.append(keys)
    
len(key_list)
    

In [ ]:
key_list[0]

In [ ]:
df['keywords'] = key_list

In [ ]:
df.head()

In [ ]:
count_list = []
key_words=''
keyword_list = df['keywords']
combine_keyword = [[' '.join(i)] for i in keyword_list]
df['combine_keywords'] = combine_keyword

In [ ]:
df.head()

In [ ]:
data = df['combine_keywords']
result = []
for sublist in data:
    for item in sublist:
        result.append(item)

df['combine_keywords'] = result
df.head()

# Input Data

In [ ]:
import numpy as np
input_data = np.array(df['combine_keywords'])
input_data = input_data.flatten()

In [ ]:
input_data.shape

# Output Data

In [ ]:
output_data = df['cluster']

In [ ]:
output_data.shape

# Train test split

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split (input_data,output_data,test_size=0.20,random_state=1,stratify=output_data)
y_train.shape

# Text to numerical conversion

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
train_data=vectorizer.fit_transform(x_train)
test_data=vectorizer.transform(x_test)
train_data

In [ ]:
train_data.shape

In [ ]:
from sklearn.decomposition import TruncatedSVD
def svd_tfidf_matrix(matrix):
    svd = TruncatedSVD(n_components=3)
    return svd.fit_transform(matrix)

In [ ]:
train_data = svd_tfidf_matrix(matrix=train_data)

In [ ]:
test_data = svd_tfidf_matrix(matrix=test_data)

In [ ]:
train_data.shape

In [ ]:
test_data.shape

# XGB Classifier

In [ ]:
#!pip install xgboost --upgrade
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# fit model no training data
model = XGBClassifier()
model.fit(train_data, y_train)
print(model)

In [ ]:
import xgboost 
print(xgboost.__version__)

In [ ]:
print(model.score(train_data, y_train))
print(model.score(test_data,y_test))

In [ ]:
test_data[0]

In [ ]:
from sklearn.metrics import auc, accuracy_score, confusion_matrix, mean_squared_error
y_pred = model.predict(test_data)

mse=mean_squared_error(y_test, y_pred)

print(np.sqrt(mse))

# Save the model

In [ ]:
# save the model to disk
import pickle
filename = 'xgboost.pb'
pickle.dump(model, open(filename, 'wb'))

# Test Input Keyword

In [ ]:
keyword = "standup comedy videos"

In [ ]:
keyword_data=np.array(keyword)
keyword_data = keyword_data.flatten()
keyword_vect=vectorizer.fit_transform(keyword_data)
keyword_vect

# Load the saved ML Model

In [ ]:
# load the model from disk
loaded_model = pickle.load(open(filename, 'rb'))
result = loaded_model.predict(keyword_vect)
print(result)

In [ ]:
result_cluster = df.loc[df['cluster'] == result[0]]
result_cluster

In [ ]:
result_cluster['viewCount'] = result_cluster['viewCount'].astype('int32')

In [ ]:
result_cluster['viewCount'].dtype

In [ ]:
#Ten largest values in view Count column
recommended_data = result_cluster.nlargest(10,['viewCount'])
recommended_data

In [ ]:
recommendation_videos = recommended_data[['ContentId','title']]
recommendation_videos

In [ ]:
keyword